In [1]:
!pip install -U datasets
# Install dependencies
!pip install autogluon
!pip install -U sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

In [2]:
from datasets import load_dataset

# Load the dataset directly from Hugging Face
dataset = load_dataset("TimSchopf/arxiv_categories")

# Convert each split to pandas
df_train = dataset['train'].to_pandas()
df_test = dataset['test'].to_pandas()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/12.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/12.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/163168 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20396 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20397 [00:00<?, ? examples/s]

In [3]:
!ls -lh


total 4.0K
drwxr-xr-x 1 root root 4.0K Jul  7 19:55 sample_data


In [4]:
# Map to 8 major categories
category_mapping = {
    "cs": "Computer Science",
    "math": "Mathematics",
    "physics": "Physics",
    "astro-ph": "Astrophysics",
    "cond-mat": "Condensed Matter",
    "quant-ph": "Quantum Physics",
    "stat": "Statistics",
    "eess": "Electrical Engineering"
}

# Handle various formats of categories
def map_label(category_entry):
    # Ensure it's a list
    if isinstance(category_entry, (list, tuple, set)):
        category_list = list(category_entry)
    elif isinstance(category_entry, str):
        category_list = [category_entry]
    elif hasattr(category_entry, "tolist"):  # NumPy array
        category_list = category_entry.tolist()
    else:
        return "Other"

    # Extract and map the primary category
    if len(category_list) > 0:
        primary = category_list[0].split("->")[-1]
        for key in category_mapping:
            if primary.startswith(key):
                return category_mapping[key]
    return "Other"

# Apply mapping
df_train["label"] = df_train["categories"].apply(map_label)
df_test["label"] = df_test["categories"].apply(map_label)


In [5]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

X_train = model.encode(df_train["title"].tolist(), batch_size=64, show_progress_bar=True, device=device)
X_test = model.encode(df_test["title"].tolist(), batch_size=64, show_progress_bar=True, device=device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2550 [00:00<?, ?it/s]

Batches:   0%|          | 0/319 [00:00<?, ?it/s]

In [6]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(df_train["label"])
y_test = le.transform(df_test["label"])


In [7]:
import pandas as pd


# Prepare DataFrames for AutoGluon
train_df = pd.DataFrame(X_train)
train_df['label'] = y_train

test_df = pd.DataFrame(X_test)
test_df['label'] = y_test

# Train with AutoGluon
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(label="label", eval_metric="accuracy").fit(
    train_df,
    time_limit=600,
    presets='good_quality',
    hyperparameters={
        'GBM': {},
        'CAT': {},
        'XGB': {},
        'RF': {},
        'XT': {}
    }
)



No path specified. Models will be saved in: "AutogluonModels/ag-20250709_092101"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          2
Memory Avail:       9.36 GB / 12.67 GB (73.8%)
Disk Space Avail:   69.08 GB / 112.64 GB (61.3%)
Presets specified: ['good_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack is enab

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ✅ Evaluate model
performance = predictor.evaluate(test_df)
print("Evaluation Metrics:", performance)
print("Accuracy:", performance.get("accuracy", "N/A"))

# ✅ Predict and plot confusion matrix
y_true = test_df["label"]
y_pred = predictor.predict(test_df.drop(columns=["label"]))

# Optional: If you used LabelEncoder, and have le.classes_
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder().fit(y_train)
# display_labels = le.classes_

# If you don't have le, use this instead:
display_labels = sorted(y_true.unique())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=display_labels)
disp.plot(xticks_rotation=45)
plt.title("Confusion Matrix (AutoGluon)")
plt.tight_layout()
plt.show()

# ✅ Plot label distribution in training set
label_counts = y_train.value_counts()
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.xticks(rotation=45)
plt.title("Label Distribution (Train)")
plt.ylabel("Count")
plt.xlabel("Class")
plt.tight_layout()
plt.show()


NameError: name 'predictor' is not defined

In [1]:
from sklearn.metrics import classification_report

# 🔍 Evaluate with classification metrics
y_test = test_df['label']
y_pred = predictor.predict(test_df.drop(columns=['label']))
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))


NameError: name 'test_df' is not defined